# 04 Interpretability

**Project:** Interpretable vs Black-Box Models for Predicting Death Events in Heart Failure Patients

This notebook explains the models after training.

**Salem Part**
- Logistic Regression coefficients
- Random Forest feature importance
- Random Forest permutation importance

**Tala Part**
- Decision Tree interpretation
- Gradient Boosting feature importance

I am keeping the same with-time and without-time setup from the proposal so the interpretation matches the model results.

## What this notebook saves

This notebook saves tables and figures that can be used later in `05_final_results.ipynb` and in the final paper.

Saved tables go to:

```text
results/tables/
```

Saved figures go to:

```text
results/figures/
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TARGET_COL = "DEATH_EVENT"

In [ ]:
# This lets the notebook work whether I run it from the main folder or from the notebooks folder.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Tables folder:", TABLES_DIR)
print("Figures folder:", FIGURES_DIR)

## Load the preprocessed files

These files should already exist from `02_preprocessing.ipynb`.

I am using the same files from the model notebook so the interpretation matches the models exactly.

In [ ]:
def load_table(file_name: str) -> pd.DataFrame:
    file_path = TABLES_DIR / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"Missing file: {file_path}")

    table = pd.read_csv(file_path)
    table = table.loc[:, ~table.columns.str.contains(r"^Unnamed")]
    return table


def load_target(file_name: str) -> pd.Series:
    target_table = load_table(file_name)

    if TARGET_COL in target_table.columns:
        target = target_table[TARGET_COL]
    elif target_table.shape[1] == 1:
        target = target_table.iloc[:, 0]
    else:
        raise ValueError(f"Could not find the target column in {file_name}")

    return target.astype(int)

In [ ]:
# Salem Part files
X_train_lr_with_time = load_table("X_train_lr_with_time.csv")
X_test_lr_with_time = load_table("X_test_lr_with_time.csv")
X_train_lr_without_time = load_table("X_train_lr_without_time.csv")
X_test_lr_without_time = load_table("X_test_lr_without_time.csv")

X_train_rf_with_time = load_table("X_train_rf_with_time.csv")
X_test_rf_with_time = load_table("X_test_rf_with_time.csv")
X_train_rf_without_time = load_table("X_train_rf_without_time.csv")
X_test_rf_without_time = load_table("X_test_rf_without_time.csv")

y_train = load_target("y_train.csv")
y_test = load_target("y_test.csv")

print("Loaded Salem files successfully.")
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Quick checks before interpreting anything.
assert len(X_train_lr_with_time) == len(y_train)
assert len(X_test_lr_with_time) == len(y_test)
assert len(X_train_lr_without_time) == len(y_train)
assert len(X_test_lr_without_time) == len(y_test)

assert len(X_train_rf_with_time) == len(y_train)
assert len(X_test_rf_with_time) == len(y_test)
assert len(X_train_rf_without_time) == len(y_train)
assert len(X_test_rf_without_time) == len(y_test)

assert "time" in X_train_lr_with_time.columns
assert "time" not in X_train_lr_without_time.columns
assert "time" in X_train_rf_with_time.columns
assert "time" not in X_train_rf_without_time.columns

print("Checks passed. The files line up correctly.")

# Salem Part

## Re-train the two models for interpretation

I am re-training the same two models from `03_models.ipynb` here because this notebook needs the fitted models to pull coefficients and feature importances.

In [ ]:
logistic_with_time = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

logistic_without_time = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

random_forest_with_time = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    min_samples_leaf=2,
)

random_forest_without_time = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    min_samples_leaf=2,
)

logistic_with_time.fit(X_train_lr_with_time, y_train)
logistic_without_time.fit(X_train_lr_without_time, y_train)
random_forest_with_time.fit(X_train_rf_with_time, y_train)
random_forest_without_time.fit(X_train_rf_without_time, y_train)

print("Models fitted for interpretation.")

## Logistic Regression coefficients

For Logistic Regression, the coefficient tells me the direction of the relationship with the predicted risk of `DEATH_EVENT = 1`.

- Positive coefficient: the feature pushes the model more toward predicting death event.
- Negative coefficient: the feature pushes the model more away from predicting death event.

The continuous variables were scaled during preprocessing, so the coefficients are easier to compare than raw unscaled measurements.

In [ ]:
def make_logistic_coefficient_table(model, feature_names, time_version: str) -> pd.DataFrame:
    coef_table = pd.DataFrame({
        "feature": feature_names,
        "coefficient": model.coef_[0],
    })

    coef_table["absolute_coefficient"] = coef_table["coefficient"].abs()
    coef_table["direction"] = np.where(
        coef_table["coefficient"] > 0,
        "higher predicted risk",
        "lower predicted risk",
    )
    coef_table["time_version"] = time_version

    coef_table = coef_table.sort_values("absolute_coefficient", ascending=False).reset_index(drop=True)
    return coef_table


logistic_coef_with_time = make_logistic_coefficient_table(
    logistic_with_time,
    X_train_lr_with_time.columns,
    "with time",
)

logistic_coef_without_time = make_logistic_coefficient_table(
    logistic_without_time,
    X_train_lr_without_time.columns,
    "without time",
)

logistic_coef_with_time.to_csv(TABLES_DIR / "logistic_coefficients_with_time.csv", index=False)
logistic_coef_without_time.to_csv(TABLES_DIR / "logistic_coefficients_without_time.csv", index=False)

logistic_coef_with_time

In [ ]:
logistic_coef_without_time

In [ ]:
def plot_logistic_coefficients(coef_table: pd.DataFrame, title: str, file_name: str, top_n: int = 12):
    plot_df = coef_table.head(top_n).copy()
    plot_df = plot_df.sort_values("coefficient")

    plt.figure(figsize=(9, 6))
    plt.barh(plot_df["feature"], plot_df["coefficient"])
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.title(title)
    plt.xlabel("Logistic Regression coefficient")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / file_name, dpi=300, bbox_inches="tight")
    plt.show()


plot_logistic_coefficients(
    logistic_coef_with_time,
    "Logistic Regression Coefficients With Time",
    "logistic_coefficients_with_time.png",
)

plot_logistic_coefficients(
    logistic_coef_without_time,
    "Logistic Regression Coefficients Without Time",
    "logistic_coefficients_without_time.png",
)

## Random Forest feature importance

For Random Forest, I first use the built-in feature importance values.

This tells me which features the forest used the most when making splits across the trees. A larger value means the feature was more important inside the Random Forest model.

In [ ]:
def make_random_forest_importance_table(model, feature_names, time_version: str) -> pd.DataFrame:
    importance_table = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    })

    importance_table["time_version"] = time_version
    importance_table = importance_table.sort_values("importance", ascending=False).reset_index(drop=True)
    return importance_table


rf_importance_with_time = make_random_forest_importance_table(
    random_forest_with_time,
    X_train_rf_with_time.columns,
    "with time",
)

rf_importance_without_time = make_random_forest_importance_table(
    random_forest_without_time,
    X_train_rf_without_time.columns,
    "without time",
)

rf_importance_with_time.to_csv(TABLES_DIR / "random_forest_feature_importance_with_time.csv", index=False)
rf_importance_without_time.to_csv(TABLES_DIR / "random_forest_feature_importance_without_time.csv", index=False)

rf_importance_with_time

In [ ]:
rf_importance_without_time

In [ ]:
def plot_feature_importance(importance_table: pd.DataFrame, value_col: str, title: str, file_name: str, top_n: int = 12):
    plot_df = importance_table.head(top_n).copy()
    plot_df = plot_df.sort_values(value_col)

    plt.figure(figsize=(9, 6))
    plt.barh(plot_df["feature"], plot_df[value_col])
    plt.title(title)
    plt.xlabel(value_col.replace("_", " ").title())
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / file_name, dpi=300, bbox_inches="tight")
    plt.show()


plot_feature_importance(
    rf_importance_with_time,
    "importance",
    "Random Forest Feature Importance With Time",
    "random_forest_feature_importance_with_time.png",
)

plot_feature_importance(
    rf_importance_without_time,
    "importance",
    "Random Forest Feature Importance Without Time",
    "random_forest_feature_importance_without_time.png",
)

## Random Forest permutation importance

I also calculate permutation importance on the test set.

This checks what happens when one feature is shuffled. If shuffling a feature hurts the model a lot, then that feature is important for prediction.

I am using this as a second check because built-in Random Forest importance can sometimes favor certain types of features.

In [ ]:
def make_permutation_importance_table(model, X_test, y_test, time_version: str) -> pd.DataFrame:
    perm_result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
    )

    perm_table = pd.DataFrame({
        "feature": X_test.columns,
        "permutation_importance_mean": perm_result.importances_mean,
        "permutation_importance_std": perm_result.importances_std,
    })

    perm_table["time_version"] = time_version
    perm_table = perm_table.sort_values("permutation_importance_mean", ascending=False).reset_index(drop=True)
    return perm_table


rf_perm_with_time = make_permutation_importance_table(
    random_forest_with_time,
    X_test_rf_with_time,
    y_test,
    "with time",
)

rf_perm_without_time = make_permutation_importance_table(
    random_forest_without_time,
    X_test_rf_without_time,
    y_test,
    "without time",
)

rf_perm_with_time.to_csv(TABLES_DIR / "random_forest_permutation_importance_with_time.csv", index=False)
rf_perm_without_time.to_csv(TABLES_DIR / "random_forest_permutation_importance_without_time.csv", index=False)

rf_perm_with_time

In [ ]:
rf_perm_without_time

In [ ]:
plot_feature_importance(
    rf_perm_with_time,
    "permutation_importance_mean",
    "Random Forest Permutation Importance With Time",
    "random_forest_permutation_importance_with_time.png",
)

plot_feature_importance(
    rf_perm_without_time,
    "permutation_importance_mean",
    "Random Forest Permutation Importance Without Time",
    "random_forest_permutation_importance_without_time.png",
)

## Salem interpretation summary

This cell creates a short text summary from the saved tables. I can use this later when writing the Results or Discussion section.

In [ ]:
def top_features_text(table: pd.DataFrame, feature_col: str, value_col: str, n: int = 5) -> str:
    top = table.head(n)
    pieces = [f"{row[feature_col]} ({row[value_col]:.4f})" for _, row in top.iterrows()]
    return ", ".join(pieces)

summary_rows = [
    {
        "section": "Logistic Regression with time",
        "top_features": top_features_text(logistic_coef_with_time, "feature", "absolute_coefficient"),
    },
    {
        "section": "Logistic Regression without time",
        "top_features": top_features_text(logistic_coef_without_time, "feature", "absolute_coefficient"),
    },
    {
        "section": "Random Forest with time",
        "top_features": top_features_text(rf_importance_with_time, "feature", "importance"),
    },
    {
        "section": "Random Forest without time",
        "top_features": top_features_text(rf_importance_without_time, "feature", "importance"),
    },
]

salem_interpretability_summary = pd.DataFrame(summary_rows)
salem_interpretability_summary.to_csv(TABLES_DIR / "salem_interpretability_summary.csv", index=False)
salem_interpretability_summary

# Tala Part

## Decision Tree interpretation

Tala can use this section for the Decision Tree model.

Possible things to include:
- tree plot
- top split features
- simple explanation of the first few splits
- comparison with and without `time`

In [ ]:
# Tala Part - Decision Tree interpretation

# Tala can load these files from results/tables/:
# X_train_tree_with_time.csv
# X_test_tree_with_time.csv
# X_train_tree_without_time.csv
# X_test_tree_without_time.csv
# y_train.csv
# y_test.csv

# Suggested output files:
# results/figures/decision_tree_with_time.png
# results/figures/decision_tree_without_time.png
# results/tables/decision_tree_feature_importance_with_time.csv
# results/tables/decision_tree_feature_importance_without_time.csv

print("Tala Part placeholder: Decision Tree interpretation goes here.")

## Gradient Boosting interpretation

Tala can use this section for Gradient Boosting.

Possible things to include:
- feature importance with `time`
- feature importance without `time`
- short comparison of the most important predictors

In [ ]:
# Tala Part - Gradient Boosting interpretation

# Tala can load these files from results/tables/:
# X_train_gb_with_time.csv
# X_test_gb_with_time.csv
# X_train_gb_without_time.csv
# X_test_gb_without_time.csv
# y_train.csv
# y_test.csv

# Suggested output files:
# results/figures/gradient_boosting_feature_importance_with_time.png
# results/figures/gradient_boosting_feature_importance_without_time.png
# results/tables/gradient_boosting_feature_importance_with_time.csv
# results/tables/gradient_boosting_feature_importance_without_time.csv

print("Tala Part placeholder: Gradient Boosting interpretation goes here.")

## Final check

After this notebook runs, Salem's interpretation tables and figures should be saved.

Next notebook:

```text
05_final_results.ipynb
```

In [ ]:
expected_salem_files = [
    TABLES_DIR / "logistic_coefficients_with_time.csv",
    TABLES_DIR / "logistic_coefficients_without_time.csv",
    TABLES_DIR / "random_forest_feature_importance_with_time.csv",
    TABLES_DIR / "random_forest_feature_importance_without_time.csv",
    TABLES_DIR / "random_forest_permutation_importance_with_time.csv",
    TABLES_DIR / "random_forest_permutation_importance_without_time.csv",
    TABLES_DIR / "salem_interpretability_summary.csv",
    FIGURES_DIR / "logistic_coefficients_with_time.png",
    FIGURES_DIR / "logistic_coefficients_without_time.png",
    FIGURES_DIR / "random_forest_feature_importance_with_time.png",
    FIGURES_DIR / "random_forest_feature_importance_without_time.png",
    FIGURES_DIR / "random_forest_permutation_importance_with_time.png",
    FIGURES_DIR / "random_forest_permutation_importance_without_time.png",
]

missing_files = [file_path for file_path in expected_salem_files if not file_path.exists()]

if missing_files:
    print("Some expected files are missing:")
    for file_path in missing_files:
        print("-", file_path)
else:
    print("All Salem interpretability files were saved successfully.")